# project_05_triage_pipeline — all notebooks (00→05) in one

This is a **convenience copy** that concatenates the six standalone notebooks in order so you can run the whole project top-to-bottom in a single Colab session. The individual notebooks (`00_setup.ipynb` … `05_validation_plan.ipynb`) remain in this folder and are the canonical deliverables. Sections are separated by dividers; each section keeps its own setup/`import` cells (re-running them is harmless). All synthetic numbers are still labeled `EXAMPLE_DATA`.

---

## ▶︎ Section 1 / 6 — `00_setup.ipynb`

---

# 00 · Environment Setup — De Novo Protein Design Capstone

This is the **shared setup notebook** every project starts from. Run it top to bottom
*once per Colab session*. It:

1. detects your GPU and warns if you're on a weak/absent one,
2. installs a light, pinned core toolset (Biopython, py3Dmol, foldseek-less utilities),
3. optionally installs heavier tools (ColabFold, ESMFold) on demand,
4. prints exact versions for your `LOG.md` (reproducibility is graded).

> **Compute reality.** A free Colab **T4** runs ColabFold, ESMFold, ProteinMPNN, and small
> RFdiffusion jobs. **BindCraft / RFantibody / large RFdiffusion** want an **A100** (Colab Pro+
> or a cluster). Each project's `MANUAL.md` states its tier. Don't fight a T4 to do an A100 job —
> plan your batch sizes around it.

## 1 · GPU & environment check

In [1]:
import subprocess, sys, platform, textwrap

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

print("Python :", sys.version.split()[0])
print("Platform:", platform.platform())

gpu = sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null")
if gpu:
    print("GPU    :", gpu)
    name = gpu.lower()
    if "t4" in name:
        print(textwrap.fill(
            "NOTE: T4 detected. Good for ColabFold/ESMFold/ProteinMPNN/small RFdiffusion. "
            "For BindCraft/RFantibody/large diffusion, switch to A100 (Colab Pro+) or a cluster.", 88))
    elif any(x in name for x in ("a100", "l4", "v100")):
        print("NOTE: capable GPU — heavier tools (BindCraft/RFantibody) are feasible.")
else:
    print("GPU    : NONE FOUND")
    print(textwrap.fill(
        "WARNING: No GPU. Go to Runtime → Change runtime type → Hardware accelerator → GPU. "
        "Structure prediction on CPU is impractically slow.", 88))

Python : 3.11.15
Platform: Linux-6.18.5-x86_64-with-glibc2.39
GPU    : NONE FOUND
Structure prediction on CPU is impractically slow.


## 2 · Pinned core install (fast, T4-friendly)

These are light and used across every project. Pins are conservative; bump them in your repo if needed and **log it**.

In [2]:
# Core utilities used in every project. Quiet + pinned.
%pip -q install biopython==1.84 py3Dmol==2.4.0 numpy pandas matplotlib seaborn tqdm requests 2>/dev/null
print("Core install done.")

Note: you may need to restart the kernel to use updated packages.
Core install done.


In [3]:
# Version stamp — copy this block's output into your LOG.md for reproducibility.
import importlib, datetime
mods = ["Bio", "py3Dmol", "numpy", "pandas", "matplotlib", "seaborn", "tqdm", "requests"]
print("# Environment stamp", datetime.datetime.utcnow().isoformat(timespec="seconds"), "UTC")
for m in mods:
    try:
        v = importlib.import_module(m).__version__
    except Exception:
        v = "n/a"
    print(f"{m:14s} {v}")

# Environment stamp 2026-06-24T03:14:51 UTC
Bio            1.84
py3Dmol        2.4.0


numpy          2.4.6


pandas         3.0.3
matplotlib     3.11.0


seaborn        0.13.2
tqdm           4.68.3
requests       2.33.1


## 3 · Heavy tools — install *on demand*

Don't install these unless your project needs them this session (they're slow to set up).
Each is wrapped in a function so you only pay the cost when you call it.

In [4]:
def install_colabfold():
    """ColabFold (AF2). ~3–5 min on first install. T4 OK."""
    import subprocess
    subprocess.run("pip -q install 'colabfold[alphafold-minus-jax]'", shell=True)
    # On Colab, the standard route is the localcolabfold installer or the ColabFold notebook;
    # here we expose the pip route. If it fails, fall back to the official ColabFold notebook
    # and import your sequences. Log whichever path you used.
    print("ColabFold install attempted. Verify with: from colabfold.batch import run")

def install_esmfold():
    """ESMFold via HuggingFace transformers. T4 OK for <~400 aa."""
    import subprocess
    subprocess.run("pip -q install 'transformers>=4.40' accelerate", shell=True)
    print("ESMFold deps installed. Load with transformers EsmForProteinFolding.")

print("Helpers ready: install_colabfold(), install_esmfold().")

Helpers ready: install_colabfold(), install_esmfold().


## 4 · Reproducibility helpers

Call `set_seeds()` at the top of every run, and use `log()` to append to your `LOG.md`.

In [5]:
import os, random
import numpy as np

def set_seeds(seed: int = 0):
    random.seed(seed); np.random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass
    print(f"seeds set to {seed}")

def log(msg: str, path: str = "LOG.md"):
    import datetime
    stamp = datetime.datetime.utcnow().isoformat(timespec="seconds")
    with open(path, "a") as fh:
        fh.write(f"- {stamp}Z · {msg}\n")
    print("logged:", msg)

set_seeds(0)
log("Ran 00_setup; environment stamped.")

seeds set to 0
logged: Ran 00_setup; environment stamped.


## 5 · (Optional) Mount Google Drive for persistence

Colab sessions are ephemeral. Mount Drive to keep your `results/` and design pools between sessions.

In [6]:
# from google.colab import drive
# drive.mount("/content/drive")
# WORKDIR = "/content/drive/MyDrive/denovo_capstone/project_XX"
# import os; os.makedirs(WORKDIR, exist_ok=True); os.chdir(WORKDIR)
print("Uncomment to mount Drive and set your working directory.")

Uncomment to mount Drive and set your working directory.


---
**Next:** open `01_define_and_explore.ipynb`. Keep this session alive — re-running `00_setup`
each new session is normal. Record every version and seed in `LOG.md`.

---

## ▶︎ Section 2 / 6 — `01_define_and_explore.ipynb`

---

# 01 · Define & Explore — spec the filter API + implement Layer 1

**Standard slot:** *define & explore.* **For Project 05 this means:** there is no target to explore —
you explore the **API you are building**. Walk through the `Design` dataclass and the four layer
functions in the shared engine, internalize the **discrimination problem**, then implement and verify
**Layer 1 (self-consistency)** with inline asserts on planted known-good / known-bad `EXAMPLE_DATA`
designs (D0).

Run `00_setup.ipynb` first in this session.

## The discrimination problem (read this first)

A campaign produces **thousands** of candidate designs, each with in-silico metrics. Triage means
returning a small, ranked, defensible short-list. The uncomfortable truth this whole project is built
around:

> **No in-silico metric — and no single layer — perfectly separates true hits from false ones.
> Filters _enrich_ the pool; they do not _guarantee_ a hit.**

Each layer raises the fraction of truly-good designs among the survivors (good), but also discards
some real hits (false negatives) and lets some duds through (false positives). Your deliverable is a
clean, **tested** engine *and* an honest map of where it fails — not a perfect classifier.

## Setup paths

In [7]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

paths ready; cwd = /home/user/biofx_python/denovo_protein_design_course/projects/project_05_triage_pipeline/notebooks


## The `Design` dataclass — the unit the engine scores

The shared engine operates on `fp.Design` records: one designed protein (monomer/binder/enzyme/
antibody/oligomer) plus whatever metrics your upstream predictors produced. The student's job in a
real campaign (Projects 01/03) is to *populate* these fields; this engine *scores* them. Let's read
the contract.

In [8]:
import filtering_pipeline as fp
import inspect

# The data model the whole engine revolves around:
print(inspect.getsource(fp.Design))

@dataclass
class Design:
    """One designed protein (monomer, binder complex, enzyme, ...)."""
    design_id: str
    sequence: str
    design_type: str = "monomer"          # monomer | binder | enzyme | antibody | oligomer
    # populated by the student's prediction step (notebook 01/02):
    designed_pdb: Optional[str] = None     # path to the designed backbone
    predicted_pdb: Optional[str] = None    # path to AF2/ESMFold prediction of the sequence
    plddt: Optional[float] = None          # mean pLDDT (functional region if relevant)
    plddt_catalytic: Optional[float] = None
    pae_interaction: Optional[float] = None
    scrmsd: Optional[float] = None         # filled by self_consistency() if not provided
    scrmsd_orthogonal: Optional[float] = None
    tm_to_pdb: Optional[float] = None      # novelty: <0.5 == novel fold
    solubility: Optional[float] = None     # e.g. CamSol-style score
    rosetta_dG: Optional[float] = None     # interface energy for binders (REU)
    sha

### Key fields, by which layer consumes them

| Field | Layer | Meaning |
|-------|-------|---------|
| `scrmsd`, `plddt`, `pae_interaction` | L1 self-consistency | does the sequence fold back to the design? (pLDDT is confidence, **NOT** stability) |
| `plddt_catalytic`, `catalytic_geom_rmsd` | L1 (enzymes) | active-site confidence + theozyme geometry |
| `scrmsd_orthogonal` | L2 orthogonal | a *second* predictor (ESMFold/Boltz) agrees |
| `solubility`, `rosetta_dG`, `shape_complementarity` | L3 physics | aggregation risk + (binders) interface energy & packing |
| `md_rmsd` | L4 dynamics | structure does not drift over a short MD |
| `layers_passed`, `score`, `notes` | filled by the pipeline | survival depth, composite score, audit trail |

## The layer-function signatures — the API you are building

These are the functions you implement and test. Read their signatures and contracts now; you will
verify Layer 1 below, Layers 2+3 in notebook 02, run the whole thing in 03, and analyze it in 04.

In [9]:
for fn in (fp.self_consistency, fp.orthogonal_check, fp.physics_filter,
           fp.dynamics_filter, fp.rank_designs, fp.run_pipeline, fp.report):
    sig = inspect.signature(fn)
    doc = (fn.__doc__ or "").strip().splitlines()[0]
    print(f"{fn.__name__}{sig}\n    {doc}\n")

self_consistency(d: 'Design', cutoffs: 'dict') -> 'bool'
    scRMSD + pLDDT (+ pAE for complexes). Computes scRMSD if PDBs are present.

orthogonal_check(d: 'Design', max_scrmsd: 'float' = 2.5) -> 'bool'
    A second predictor (ESMFold/Boltz) should agree with the design.

physics_filter(d: 'Design', cutoffs: 'dict', min_solubility: 'float' = -1.0) -> 'bool'
    Solubility/aggregation + (for binders) interface energy & shape complementarity.

dynamics_filter(d: 'Design', max_md_rmsd: 'float' = 3.0) -> 'bool'
    Structure should not drift far from the design over a short MD run.

rank_designs(designs: 'list[Design]') -> 'list[Design]'
    Composite score (higher = better). Tune weights per project in the notebook.

run_pipeline(designs: 'list[Design]', design_type: 'str' = 'monomer', cutoffs: 'Optional[dict]' = None, use_layers=(1, 2, 3)) -> 'pd.DataFrame'
    Run the chosen layers, rank, and return a tidy DataFrame. Records survival counts.

report(df: 'pd.DataFrame', top_n: 'int' = 2

## Cutoffs by design type — and why they differ

`DEFAULT_CUTOFFS` holds per-type thresholds. They differ because the design problems differ: an
antibody CDR loop is harder to predict than an idealized helical monomer, so its scRMSD/pLDDT bars
are looser; a binder adds interface terms (`pae`, `rosetta_dG`, `sc`) a monomer does not have.
**Justifying these is a graded task** — cite the self-consistency literature (Dauparas 2022) and your
own enrichment analysis (notebook 04).

In [10]:
print("DEFAULT_CUTOFFS (starting points from the validation-ref lineage; justify + PR changes):")
for k, v in fp.DEFAULT_CUTOFFS.items():
    print(f"  {k:9s} {v}")

DEFAULT_CUTOFFS (starting points from the validation-ref lineage; justify + PR changes):
  monomer   {'scrmsd': 2.0, 'plddt': 85, 'pae': None}
  binder    {'scrmsd': 2.5, 'plddt': 80, 'pae': 10, 'rosetta_dG': -30, 'sc': 0.6}
  enzyme    {'scrmsd': 2.0, 'plddt': 85, 'plddt_cat': 90, 'cat_geom': 0.5}
  antibody  {'scrmsd': 3.0, 'plddt': 70, 'pae': 12}
  oligomer  {'scrmsd': 2.5, 'plddt': 80, 'pae': 10}


## The planted `EXAMPLE_DATA` fixtures — your "controls"

This project's controls are **planted designs with known answers**: a known-good design that should
pass every layer, and known-bad designs each built to fail a *specific* layer. They live in
`scripts/make_example_pool.py` and drive both the pool and the unit tests, so the two stay in sync.
**These are synthetic — never present their numbers as real.**

In [11]:
from make_example_pool import known_designs

planted = {r["design_id"]: r for r in known_designs()}
print(f"{len(planted)} planted fixtures:")
for did, row in planted.items():
    print(f"  {did:34s} truth={row['truth']:4s} {row['note']}")

12 planted fixtures:
  EXAMPLE_DATA_GOOD_monomer          truth=good planted known-good: passes all layers for its type
  EXAMPLE_DATA_GOOD_binder           truth=good planted known-good: passes all layers for its type
  EXAMPLE_DATA_GOOD_enzyme           truth=good planted known-good: passes all layers for its type
  EXAMPLE_DATA_GOOD_antibody         truth=good planted known-good: passes all layers for its type
  EXAMPLE_DATA_GOOD_oligomer         truth=good planted known-good: passes all layers for its type
  EXAMPLE_DATA_BAD_L1_scrmsd         truth=bad  planted known-bad: fails L1 (scRMSD 3.4 > 2.0) — confident about the WRONG fold
  EXAMPLE_DATA_BAD_L1_plddt          truth=bad  planted known-bad: fails L1 (pLDDT 72 < 85)
  EXAMPLE_DATA_BAD_L2_orthogonal     truth=bad  planted known-bad: passes L1, fails L2 (orthogonal scRMSD 3.6 > 2.5) — predictor-specific overconfidence
  EXAMPLE_DATA_BAD_L3_solubility     truth=bad  planted known-bad: passes L1-L2, fails L3 (solubility -2.4 < -1

## Implement & verify **Layer 1 — self-consistency**

Layer 1 asks: *does the sequence fold back to the shape it was designed for?* It checks `scrmsd`,
`plddt`, (and for enzymes `plddt_catalytic` + `catalytic_geom_rmsd`, for complexes `pae`) against the
type's cutoffs. Below we build `fp.Design` objects from the planted fixtures and assert the expected
pass/fail — this is your D0 evidence.

In [12]:
def design_from_row(row):
    """Build a fresh fp.Design from a planted EXAMPLE_DATA row (drop non-Design columns)."""
    drop = {"truth", "note", "design_id"}
    fields = {k: v for k, v in row.items() if k not in drop}
    return fp.Design(design_id=row["design_id"], sequence="M", **fields)

cut_mono = fp.DEFAULT_CUTOFFS["monomer"]
cut_enz = fp.DEFAULT_CUTOFFS["enzyme"]

# Known-GOOD monomer must PASS Layer 1:
good = design_from_row(planted["EXAMPLE_DATA_GOOD_monomer"])
assert fp.self_consistency(good, cut_mono) is True, "known-good monomer should pass L1"

# Known-BAD (high scRMSD: confident about the WRONG fold) must FAIL Layer 1:
bad_scrmsd = design_from_row(planted["EXAMPLE_DATA_BAD_L1_scrmsd"])
assert fp.self_consistency(bad_scrmsd, cut_mono) is False, "high-scRMSD design must fail L1"

# Known-BAD (low pLDDT) must FAIL Layer 1:
bad_plddt = design_from_row(planted["EXAMPLE_DATA_BAD_L1_plddt"])
assert fp.self_consistency(bad_plddt, cut_mono) is False, "low-pLDDT design must fail L1"

# Known-BAD enzyme (broken catalytic geometry) must FAIL Layer 1:
bad_enz = design_from_row(planted["EXAMPLE_DATA_BAD_L1_enzyme_geom"])
assert fp.self_consistency(bad_enz, cut_enz) is False, "enzyme with bad active-site geom must fail L1"

print("Layer 1 verified on planted designs:")
print(f"  GOOD_monomer        passed L1? {good.layers_passed >= 1}  (scrmsd={good.scrmsd}, plddt={good.plddt})")
print(f"  BAD_L1_scrmsd       notes: {bad_scrmsd.notes}")
print(f"  BAD_L1_plddt        notes: {bad_plddt.notes}")
print(f"  BAD_L1_enzyme_geom  notes: {bad_enz.notes}")
print("\nAll Layer-1 asserts passed. (Fixtures are EXAMPLE_DATA — synthetic.)")

Layer 1 verified on planted designs:
  GOOD_monomer        passed L1? True  (scrmsd=1.2, plddt=92.0)
  BAD_L1_scrmsd       notes: ['failed L1 self-consistency']
  BAD_L1_plddt        notes: ['failed L1 self-consistency']
  BAD_L1_enzyme_geom  notes: ['failed L1 self-consistency']

All Layer-1 asserts passed. (Fixtures are EXAMPLE_DATA — synthetic.)


### Note: the geometry helper (`ca_rmsd`)

If you *don't* pre-compute `scrmsd`, `self_consistency()` will compute it from `designed_pdb` +
`predicted_pdb` via `fp.ca_rmsd()` (Biopython `Superimposer`, teaching-grade Cα-RMSD). In this course
the upstream projects (01/03) usually provide the number directly; the helper is there for when you
have the PDBs. We don't exercise it here (no PDBs in the synthetic pool).

In [13]:
import inspect
print(inspect.getsource(fp.ca_rmsd))

def ca_rmsd(pdb_a: str, pdb_b: str) -> float:
    """Cα RMSD (Å) between two PDBs after optimal superposition. Teaching-grade."""
    from Bio.PDB import PDBParser, Superimposer
    p = PDBParser(QUIET=True)
    a = p.get_structure("a", pdb_a)
    b = p.get_structure("b", pdb_b)
    ca_a = [r["CA"] for r in a.get_residues() if "CA" in r]
    ca_b = [r["CA"] for r in b.get_residues() if "CA" in r]
    n = min(len(ca_a), len(ca_b))
    if n == 0:
        raise ValueError("No Cα atoms found / length mismatch.")
    sup = Superimposer()
    sup.set_atoms(ca_a[:n], ca_b[:n])
    return float(sup.rms)



## D0 checklist
- [ ] One-page API spec: the `Design` fields + each layer function's signature **and contract**.
- [ ] The discrimination problem stated precisely (filters enrich, not guarantee).
- [ ] Layer 1 verified: planted known-good passes; each planted known-bad fails (printout above).
- [ ] `LOG.md` entry: what you ran, seed, outcome.

**Next:** `02_generate.ipynb` — assemble the labeled pool and implement/verify Layers 2 + 3.

---

## ▶︎ Section 3 / 6 — `02_generate.ipynb`

---

# 02 · Campaign — assemble the pool + implement Layers 2 & 3

**Standard slot:** *design campaign.* **For Project 05 this means:** there is **nothing to generate** —
your "campaign" is *assembling* a labeled design pool and *building* the next two filter layers. You
generate the clearly-synthetic `EXAMPLE_DATA` pool (so the layers are developable with no GPU), then
implement and verify **Layer 2 (orthogonal)** and **Layer 3 (physics)** with unit-test asserts on
planted known-good / known-bad designs (D2).

Run `00_setup.ipynb` first in this session.

## Setup paths

In [14]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

paths ready; cwd = /home/user/biofx_python/denovo_protein_design_course/projects/project_05_triage_pipeline/notebooks


## Version-verify the pinned upstreams (tools change!)

The engine needs no GPU, but its metrics come from upstream tools that move fast. Pin the upstream
repos and **verify they still exist** before relying on them. (`requests.head`; a non-200 means the
URL moved — update your pin and log it.)

In [15]:
import requests

# Pinned upstream metric sources (verify; pin a commit/tag/version in your repo — these change):
#   Boltz      https://github.com/jwohlwend/boltz          (pin a release tag)
#   ColabFold  https://github.com/sokrypton/ColabFold      (pin a commit)
#   PyRosetta  https://www.pyrosetta.org                   (pin the release; academic license)
PINNED = {
    "Boltz (Layer 2 / affinity)":     "https://github.com/jwohlwend/boltz",
    "ColabFold (Layer 1/2 source)":   "https://github.com/sokrypton/ColabFold",
    "PyRosetta (Layer 3 physics)":    "https://www.pyrosetta.org",
}
for name, url in PINNED.items():
    try:
        r = requests.head(url, allow_redirects=True, timeout=15)
        print(f"  [{r.status_code}] {name:32s} {url}")
    except Exception as e:  # noqa: BLE001
        print(f"  [ERR] {name:32s} {url}  ({e})")
print("\nNon-200 / error ⇒ the upstream moved; update the pin in env/requirements.txt and log it.")

  [200] Boltz (Layer 2 / affinity)       https://github.com/jwohlwend/boltz


  [200] ColabFold (Layer 1/2 source)     https://github.com/sokrypton/ColabFold


  [ERR] PyRosetta (Layer 3 physics)      https://www.pyrosetta.org  (HTTPSConnectionPool(host='www.pyrosetta.org', port=443): Max retries exceeded with url: / (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden'))))

Non-200 / error ⇒ the upstream moved; update the pin in env/requirements.txt and log it.


## 1 · Build the labeled `EXAMPLE_DATA` pool

`scripts/make_example_pool.py` writes `results/pool.csv` (~200 mixed-type designs, deterministic
seed). **Every row is synthetic** (`design_id` prefixed `EXAMPLE_DATA_`), and a hidden `truth`
column records the *planted* label for the enrichment analysis in notebook 04. The pool deliberately
contains planted known-good, layer-specific known-bad, and ambiguous-near-cutoff designs — so
enrichment is imperfect and the discrimination problem is visible.

In [16]:
import filtering_pipeline as fp
import make_example_pool as mep
import pandas as pd

df = mep.make_pool(n=200)
df.to_csv("results/pool.csv", index=False)
n_good = int((df["truth"] == "good").sum())
print(f"results/pool.csv: {len(df)} EXAMPLE_DATA designs ({n_good} planted-good, {len(df)-n_good} planted-bad)")
print("design types:", df["design_type"].value_counts().to_dict())
print("\nREMINDER: every number here is SYNTHETIC (EXAMPLE_DATA) — never report it as a real result.")
df.head(6)

results/pool.csv: 200 EXAMPLE_DATA designs (109 planted-good, 91 planted-bad)
design types: {'oligomer': 47, 'monomer': 43, 'enzyme': 40, 'binder': 38, 'antibody': 32}

REMINDER: every number here is SYNTHETIC (EXAMPLE_DATA) — never report it as a real result.


,design_id,design_type,truth,scrmsd,plddt,plddt_catalytic,pae_interaction,catalytic_geom_rmsd,scrmsd_orthogonal,solubility,rosetta_dG,shape_complementarity,md_rmsd,note
0,EXAMPLE_DATA_GOOD_monomer,monomer,good,1.2,92.0,NaN,6.0,NaN,1.4,0.6,NaN,NaN,1.5,planted known-good: passes all layers for its ...
1,EXAMPLE_DATA_GOOD_binder,binder,good,1.6,88.0,NaN,6.0,NaN,1.8,0.4,-42.0,0.72,2.0,planted known-good: passes all layers for its ...
2,EXAMPLE_DATA_GOOD_enzyme,enzyme,good,1.2,92.0,95.0,6.0,0.3,1.5,0.5,NaN,NaN,1.6,planted known-good: passes all layers for its ...
3,EXAMPLE_DATA_GOOD_antibody,antibody,good,2.2,80.0,NaN,8.0,NaN,2.1,0.3,NaN,NaN,2.2,planted known-good: passes all layers for its ...
4,EXAMPLE_DATA_GOOD_oligomer,oligomer,good,1.7,88.0,NaN,6.0,NaN,1.9,0.5,-38.0,0.68,2.0,planted known-good: passes all layers for its ...
5,EXAMPLE_DATA_BAD_L1_scrmsd,monomer,bad,3.4,90.0,NaN,6.0,NaN,1.5,0.5,NaN,NaN,1.5,planted known-bad: fails L1 (scRMSD 3.4 > 2.0)...


## 2 · Implement & verify **Layer 2 — orthogonal agreement**

A design that *one* predictor (AF2) loves can still be wrong — predictors share blind spots and can be
overconfident on de novo sequences. Layer 2 requires a **second, independent** predictor (ESMFold or
Boltz, different inductive bias) to also reproduce the design: `scrmsd_orthogonal ≤ max_scrmsd`. The
planted `BAD_L2_orthogonal` design passes L1 but fails here — exactly the overconfidence case.

In [17]:
planted = {r["design_id"]: r for r in mep.known_designs()}

def design_from_row(row):
    drop = {"truth", "note", "design_id"}
    fields = {k: v for k, v in row.items() if k not in drop}
    return fp.Design(design_id=row["design_id"], sequence="M", **fields)

cut_mono = fp.DEFAULT_CUTOFFS["monomer"]

good = design_from_row(planted["EXAMPLE_DATA_GOOD_monomer"])
assert fp.orthogonal_check(good) is True, "known-good should pass L2"

bad2 = design_from_row(planted["EXAMPLE_DATA_BAD_L2_orthogonal"])
assert fp.self_consistency(bad2, cut_mono) is True, "fixture is meant to PASS L1 (AF2 likes it)"
assert fp.orthogonal_check(bad2) is False, "orthogonal disagreement must FAIL L2"

print("Layer 2 verified:")
print(f"  GOOD_monomer         orthogonal scRMSD={good.scrmsd_orthogonal} -> pass")
print(f"  BAD_L2_orthogonal    orthogonal scRMSD={bad2.scrmsd_orthogonal} -> fail  notes={bad2.notes}")

Layer 2 verified:
  GOOD_monomer         orthogonal scRMSD=1.4 -> pass
  BAD_L2_orthogonal    orthogonal scRMSD=3.6 -> fail  notes=['failed L2 orthogonal']


## 3 · Implement & verify **Layer 3 — physics**

Self-consistency and orthogonal agreement say nothing about **solubility** or **interface strength**.
Layer 3 checks a CamSol-style `solubility` score and, for binders, interface energy (`rosetta_dG`,
REU) + shape complementarity (`sc`). Keep PyRosetta / FreeBindCraft / CamSol **behind the boundary** —
the layer thresholds numbers; the notebooks (on Colab) produce them. We verify two planted bads: an
aggregation-prone monomer and a weak-interface binder.

In [18]:
cut_bind = fp.DEFAULT_CUTOFFS["binder"]

# Known-good monomer + binder pass L3:
for dt in ("monomer", "binder"):
    d = design_from_row(planted[f"EXAMPLE_DATA_GOOD_{dt}"])
    assert fp.physics_filter(d, fp.DEFAULT_CUTOFFS[dt]) is True, f"known-good {dt} should pass L3"

# Aggregation-prone monomer fails L3 on solubility:
bad_sol = design_from_row(planted["EXAMPLE_DATA_BAD_L3_solubility"])
assert fp.physics_filter(bad_sol, cut_mono) is False, "aggregation-prone monomer must fail L3"

# Weak-interface binder fails L3 on rosetta_dG / shape complementarity:
bad_int = design_from_row(planted["EXAMPLE_DATA_BAD_L3_interface"])
assert fp.physics_filter(bad_int, cut_bind) is False, "weak-interface binder must fail L3"

print("Layer 3 verified:")
print(f"  BAD_L3_solubility   solubility={bad_sol.solubility} -> fail  notes={bad_sol.notes}")
print(f"  BAD_L3_interface    rosetta_dG={bad_int.rosetta_dG}, sc={bad_int.shape_complementarity} -> fail")

Layer 3 verified:
  BAD_L3_solubility   solubility=-2.4 -> fail  notes=['failed L3 physics']
  BAD_L3_interface    rosetta_dG=-12.0, sc=0.45 -> fail


## 4 · Run the test suite (the real D2 artifact)

The asserts above are spot checks; `scripts/test_filtering.py` is the durable contract. It imports
the **shared** `filtering_pipeline` and asserts every planted case across all four layers + the
orchestration. Run it here; it must exit 0. Re-run it after **any** cutoff change — a layer with no
failing planted case is untested.

In [19]:
import subprocess, sys
res = subprocess.run([sys.executable, "../scripts/test_filtering.py"],
                     capture_output=True, text=True)
print(res.stdout)
if res.returncode != 0:
    print("STDERR:\n", res.stderr)
print("exit code:", res.returncode, "(0 = all tests passed)")

  PASS  test_layer1_bad_plddt_fails
  PASS  test_layer1_bad_scrmsd_fails
  PASS  test_layer1_enzyme_geom_fails
  PASS  test_layer1_good_passes_per_type
  PASS  test_layer2_bad_orthogonal_fails
  PASS  test_layer2_good_passes
  PASS  test_layer3_bad_interface_fails
  PASS  test_layer3_bad_solubility_fails
  PASS  test_layer3_good_passes
  PASS  test_layer4_bad_dynamics_fails
  PASS  test_layer4_good_passes
  PASS  test_ranking_orders_good_above_bad
Total designs: 2
  L1 survivors: 1  (50.0%)
  L2 survivors: 1  (50.0%)
  L3 survivors: 1  (50.0%)
  PASS  test_run_pipeline_survival_and_report

13/13 tests passed.
All tests passed against the SHARED filtering_pipeline. (All fixtures are EXAMPLE_DATA — synthetic, not real results.)

exit code: 0 (0 = all tests passed)


## D2 checklist
- [ ] `results/pool.csv`: labeled `EXAMPLE_DATA` pool (mixed types, planted + ambiguous), deterministic.
- [ ] Layer 2 (orthogonal) + Layer 3 (physics) implemented & verified on planted known-good/known-bad.
- [ ] `python scripts/test_filtering.py` exits 0 (per layer **and** per design type).
- [ ] Design log: every cutoff choice + seed + the test command + outcome, in `LOG.md`.
- [ ] 3–4 page interim report.

**Next:** `03_filter_and_rank.ipynb` — run the **shared** engine on the pool.

---

## ▶︎ Section 4 / 6 — `03_filter_and_rank.ipynb`

---

# 03 · Filter & Rank — run the shared multi-layer engine

**Standard slot:** *filter & rank* via `shared/filtering_pipeline.py` — the same module all 25
projects use. **For Project 05** you are both its *author* (you implemented the layers) and a *user*
(you run it on a pool). The point of this notebook: import the **shared** engine, build `fp.Design`
objects from the pool, run `fp.run_pipeline(...)`, and `fp.report(...)` the survival-at-each-layer
funnel + ranked CSV (D3 part 1).

> **Do not fork the module into this project.** Iterate locally against `shared/filtering_pipeline.py`
> and PR improvements back (notebook 05). This notebook *imports* it.

Run `00`–`02` first so `results/pool.csv` exists.

## Setup paths

In [20]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

paths ready; cwd = /home/user/biofx_python/denovo_protein_design_course/projects/project_05_triage_pipeline/notebooks


## Load the shared filtering pipeline
This is the cohort's shared module — improvements here are pull-requested back for everyone.

In [21]:
import filtering_pipeline as fp
import pandas as pd

print("Loaded shared filtering_pipeline from:", fp.__file__)
print("DEFAULT_CUTOFFS:")
for k, v in fp.DEFAULT_CUTOFFS.items():
    print(" ", k, v)

Loaded shared filtering_pipeline from: /home/user/biofx_python/denovo_protein_design_course/shared/filtering_pipeline.py
DEFAULT_CUTOFFS:
  monomer {'scrmsd': 2.0, 'plddt': 85, 'pae': None}
  binder {'scrmsd': 2.5, 'plddt': 80, 'pae': 10, 'rosetta_dG': -30, 'sc': 0.6}
  enzyme {'scrmsd': 2.0, 'plddt': 85, 'plddt_cat': 90, 'cat_geom': 0.5}
  antibody {'scrmsd': 3.0, 'plddt': 70, 'pae': 12}
  oligomer {'scrmsd': 2.5, 'plddt': 80, 'pae': 10}


## Build `Design` objects from the pool

The engine operates on `fp.Design` records. Map each pool row onto its fields. The synthetic pool
pre-populates the metrics; in a real campaign these come from your upstream predictions (Projects
01/03). We keep the hidden `truth` label in `extra` so notebook 04 can measure enrichment — the
engine itself never sees it.

In [22]:
import os
if not os.path.exists("results/pool.csv"):
    # regenerate if a fresh session lost it (deterministic seed)
    import make_example_pool as mep
    mep.make_pool(200).to_csv("results/pool.csv", index=False)

pool = pd.read_csv("results/pool.csv")

DESIGN_FIELDS = ["scrmsd", "plddt", "plddt_catalytic", "pae_interaction",
                 "catalytic_geom_rmsd", "scrmsd_orthogonal", "solubility",
                 "rosetta_dG", "shape_complementarity", "md_rmsd"]

def row_to_design(r):
    kw = {f: (None if pd.isna(r[f]) else float(r[f])) for f in DESIGN_FIELDS}
    return fp.Design(design_id=str(r["design_id"]), sequence="M",
                     design_type=str(r["design_type"]),
                     extra={"truth": r["truth"]}, **kw)

designs = [row_to_design(r) for _, r in pool.iterrows()]
print(len(designs), "Design objects built;",
      "types:", pool["design_type"].value_counts().to_dict())

200 Design objects built; types: {'oligomer': 47, 'monomer': 43, 'enzyme': 40, 'binder': 38, 'antibody': 32}


## Run the pipeline (per design type)

`run_pipeline()` applies the chosen layers in order and returns a ranked DataFrame with survival
counts in `df.attrs`. The pool is **mixed**, and cutoffs differ by type, so we run each design type
with its own cutoffs, then concatenate. We use Layers 1–3 here (Layer 4 / MD is optional and added in
notebook 05).

In [23]:
import pandas as pd

frames = []
survival_by_type = {}
for dt, sub in pool.groupby("design_type"):
    sub_designs = [row_to_design(r) for _, r in sub.iterrows()]
    df_dt = fp.run_pipeline(sub_designs, design_type=dt, use_layers=(1, 2, 3))
    survival_by_type[dt] = df_dt.attrs["survival"]
    df_dt["design_type"] = dt
    frames.append(df_dt)

ranked = pd.concat(frames, ignore_index=True).sort_values(
    ["layers_passed", "score"], ascending=False).reset_index(drop=True)
ranked.to_csv("results/pool_ranked.csv", index=False)
print("wrote results/pool_ranked.csv", ranked.shape)
print("\nsurvival by design type (L1->L3):")
for dt, s in survival_by_type.items():
    print(f"  {dt:9s} {s}")
ranked.head(10)[["design_id", "design_type", "layers_passed", "score",
                 "scrmsd", "plddt", "scrmsd_orthogonal", "solubility"]]

wrote results/pool_ranked.csv (200, 20)

survival by design type (L1->L3):
  antibody  {'L1': 19, 'L2': 16, 'L3': 16}
  binder    {'L1': 24, 'L2': 24, 'L3': 22}
  enzyme    {'L1': 22, 'L2': 21, 'L3': 21}
  monomer   {'L1': 30, 'L2': 28, 'L3': 25}
  oligomer  {'L1': 29, 'L2': 26, 'L3': 26}


,design_id,design_type,layers_passed,score,scrmsd,plddt,scrmsd_orthogonal,solubility
0,EXAMPLE_DATA_0060,oligomer,3,4.4148,1.150,91.663,2.133,0.050
1,EXAMPLE_DATA_0176,binder,3,4.4133,0.996,93.093,1.807,0.690
2,EXAMPLE_DATA_0170,binder,3,4.0765,1.204,85.498,1.810,0.073
3,EXAMPLE_DATA_0094,binder,3,4.0723,1.326,90.437,1.337,0.105
4,EXAMPLE_DATA_0049,binder,3,4.0516,1.287,84.166,1.947,-0.267
5,EXAMPLE_DATA_0059,binder,3,4.0098,1.385,86.699,1.889,0.685
6,EXAMPLE_DATA_0121,binder,3,3.9980,1.566,86.330,1.713,0.325
7,EXAMPLE_DATA_0027,oligomer,3,3.9633,1.435,85.314,1.758,0.224
8,EXAMPLE_DATA_0090,binder,3,3.9508,1.568,90.125,1.811,0.476
9,EXAMPLE_DATA_0018,binder,3,3.9454,1.569,89.691,2.146,0.434


## Survival-at-each-layer via `report()`

`report()` prints the hit-rate accounting and draws the survival funnel. Here we report the **whole
pool** treated with monomer cutoffs for a single, comparable funnel figure (the per-type run above is
the rigorous version). Read the bars as a funnel: steep drops show which layer discriminates; a layer
that cuts nothing is too lenient or redundant.

In [24]:
import matplotlib
matplotlib.use("Agg")  # headless-safe; Colab will still display inline

all_designs = [row_to_design(r) for _, r in pool.iterrows()]
df_all = fp.run_pipeline(all_designs, design_type="monomer", use_layers=(1, 2, 3))
top = fp.report(df_all, top_n=15, save_prefix="results/p05")
print("\nsaved results/p05_survival.png + results/p05_ranked.csv")
top

Total designs: 200
  L1 survivors: 102  (51.0%)
  L2 survivors: 95  (47.5%)
  L3 survivors: 87  (43.5%)



saved results/p05_survival.png + results/p05_ranked.csv


,design_id,design_type,layers_passed,score,scrmsd,plddt,pae_interaction,rosetta_dG,tm_to_pdb
0,EXAMPLE_DATA_0060,monomer,3,4.4148,1.150,91.663,4.170,-37.283,None
1,EXAMPLE_DATA_0176,monomer,3,4.4133,0.996,93.093,6.097,-36.823,None
2,EXAMPLE_DATA_0170,monomer,3,4.0765,1.204,85.498,6.678,-36.603,None
3,EXAMPLE_DATA_0094,monomer,3,4.0723,1.326,90.437,7.950,-45.323,None
4,EXAMPLE_DATA_0059,monomer,3,4.0098,1.385,86.699,6.703,-41.818,None
5,EXAMPLE_DATA_0121,monomer,3,3.9980,1.566,86.330,4.635,-41.847,None
6,EXAMPLE_DATA_0027,monomer,3,3.9633,1.435,85.314,6.113,-40.231,None
7,EXAMPLE_DATA_0090,monomer,3,3.9508,1.568,90.125,5.733,-42.266,None
8,EXAMPLE_DATA_0018,monomer,3,3.9454,1.569,89.691,5.519,-41.369,None
9,EXAMPLE_DATA_0172,monomer,3,3.9426,1.551,90.521,5.406,-39.445,None


## Honest survival accounting

How many designs reach each depth? `layers_passed` records the deepest layer each design survived.
This is the standard cohort artifact — but remember it is *survival*, not *correctness*. Notebook 04
adds the labeled-pool **enrichment** view (how many survivors are actually good).

In [25]:
print("layers_passed distribution (whole pool, monomer cutoffs):")
print(df_all["layers_passed"].value_counts().sort_index())
n = len(df_all)
surv = df_all.attrs["survival"]
print("\nfunnel:")
print(f"  total            {n}")
for layer, k in surv.items():
    print(f"  survived {layer}      {k}  ({100*k/n:.1f}%)")

layers_passed distribution (whole pool, monomer cutoffs):
layers_passed
0    98
1     7
2     8
3    87
Name: count, dtype: int64

funnel:
  total            200
  survived L1      102  (51.0%)
  survived L2      95  (47.5%)
  survived L3      87  (43.5%)


## D3 (part 1) checklist
- [ ] `results/pool_ranked.csv` produced by the **shared** module (not a one-off script).
- [ ] Survival-at-each-layer reported (funnel figure `results/p05_survival.png`).
- [ ] Per-design-type run done (each type with its own cutoffs).
- [ ] Mapping assumptions (which fields → which `Design` attributes) written down.

**Next:** `04_validate.ipynb` — the discrimination-problem analysis (enrichment + cutoff sensitivity).

---

## ▶︎ Section 5 / 6 — `04_validate.ipynb`

---

# 04 · Validate — the discrimination-problem analysis

**Standard slot:** *validate (in silico).* **For Project 05 this is the core science:** using the
hidden `truth` labels in the `EXAMPLE_DATA` pool, measure the **enrichment** each layer buys
(precision / recall), run a **cutoff-sensitivity sweep**, and show honestly that **no single metric
or layer cleanly separates true from false** (D3 part 2).

> Every number and figure here is `EXAMPLE_DATA` — **synthetic, for illustrating the method**, never a
> real result. On a real labeled pool (from a design paper with experimental outcomes) the same code
> produces a real analysis.

Needs `results/pool.csv` with the `truth` column.

## Setup paths

In [26]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

paths ready; cwd = /home/user/biofx_python/denovo_protein_design_course/projects/project_05_triage_pipeline/notebooks


## 1 · Enrichment at each layer (precision & recall)

For each layer depth we ask: of the designs that survive to here, what fraction are *truly* good
(**precision** = enrichment), and what fraction of *all* truly-good designs do we still retain
(**recall**)? A good layer raises precision; watch the recall you pay for it. We run the layers and
read `layers_passed` against the planted `truth`.

In [27]:
import filtering_pipeline as fp
import make_example_pool as mep
import pandas as pd, numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import os

if not os.path.exists("results/pool.csv"):
    mep.make_pool(200).to_csv("results/pool.csv", index=False)
pool = pd.read_csv("results/pool.csv")

DESIGN_FIELDS = ["scrmsd", "plddt", "plddt_catalytic", "pae_interaction",
                 "catalytic_geom_rmsd", "scrmsd_orthogonal", "solubility",
                 "rosetta_dG", "shape_complementarity", "md_rmsd"]

def row_to_design(r):
    kw = {f: (None if pd.isna(r[f]) else float(r[f])) for f in DESIGN_FIELDS}
    return fp.Design(design_id=str(r["design_id"]), sequence="M",
                     design_type=str(r["design_type"]), extra={"truth": r["truth"]}, **kw)

# Run layers per design type (own cutoffs), then collect layers_passed + truth.
recs = []
for dt, sub in pool.groupby("design_type"):
    ds = [row_to_design(r) for _, r in sub.iterrows()]
    fp.run_pipeline(ds, design_type=dt, use_layers=(1, 2, 3, 4))
    for d in ds:
        recs.append(dict(design_id=d.design_id, design_type=dt,
                         layers_passed=d.layers_passed, truth=d.extra["truth"]))
res = pd.DataFrame(recs)
res["is_good"] = (res["truth"] == "good").astype(int)
n_good_total = int(res["is_good"].sum())
print(f"pool: {len(res)} designs, {n_good_total} truly-good (EXAMPLE_DATA labels)")

pool: 200 designs, 109 truly-good (EXAMPLE_DATA labels)


In [28]:
rows = []
total = len(res)
for depth in range(0, 5):  # survivors that passed >= depth layers
    surv = res[res["layers_passed"] >= depth] if depth > 0 else res
    n = len(surv)
    n_good = int(surv["is_good"].sum())
    precision = n_good / n if n else float("nan")          # enrichment
    recall = n_good / n_good_total if n_good_total else float("nan")
    rows.append(dict(layer=("pool" if depth == 0 else f">=L{depth}"),
                     survivors=n, good_survivors=n_good,
                     precision=round(precision, 3), recall=round(recall, 3)))
enr = pd.DataFrame(rows)
print("Enrichment by layer depth (EXAMPLE_DATA):")
print(enr.to_string(index=False))

baseline = enr.iloc[0]["precision"]
print(f"\nbaseline good-fraction = {baseline:.3f}; "
      f"deepest-survivor precision = {enr.iloc[-1]['precision']:.3f} "
      f"(enrichment, but recall fell to {enr.iloc[-1]['recall']:.3f}).")

Enrichment by layer depth (EXAMPLE_DATA):
layer  survivors  good_survivors  precision  recall
 pool        200             109      0.545   1.000
 >=L1        124             108      0.871   0.991
 >=L2        115             106      0.922   0.972
 >=L3        110             106      0.964   0.972
 >=L4        106             105      0.991   0.963

baseline good-fraction = 0.545; deepest-survivor precision = 0.991 (enrichment, but recall fell to 0.963).


In [29]:
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
ax[0].plot(enr["layer"], enr["precision"], "o-", label="precision (enrichment)")
ax[0].plot(enr["layer"], enr["recall"], "s--", label="recall")
ax[0].axhline(baseline, color="grey", ls=":", lw=0.8, label="baseline good-fraction")
ax[0].set_ylim(0, 1.02); ax[0].set_ylabel("fraction"); ax[0].set_title("Enrichment vs recall (EXAMPLE_DATA)")
ax[0].legend(fontsize=8)
ax[1].bar(enr["layer"], enr["survivors"]); ax[1].set_title("Survivors per layer (EXAMPLE_DATA)")
for i, v in enumerate(enr["survivors"]):
    ax[1].text(i, v, str(v), ha="center", va="bottom", fontsize=8)
plt.tight_layout(); plt.savefig("results/p05_enrichment.png", dpi=150); plt.show()
print("saved results/p05_enrichment.png  (SYNTHETIC EXAMPLE_DATA)")

saved results/p05_enrichment.png  (SYNTHETIC EXAMPLE_DATA)


**The point:** precision rises layer by layer (the filter *enriches*) but never reaches 1.0 —
**false positives survive every layer** — and recall falls, so **true hits are discarded**. That gap
is the discrimination problem. On real data it is usually worse, not better.

## 2 · No single metric separates true from false

Plot each metric's distribution split by `truth`. The good/bad distributions **overlap** — there is no
threshold that cleanly separates them. (We show ROC-AUC per metric only as a separability summary; on
this synthetic pool it is illustrative, not a result.)

In [30]:
from sklearn.metrics import roc_auc_score

metric_dir = {"scrmsd": -1, "plddt": +1, "pae_interaction": -1,
              "scrmsd_orthogonal": -1, "solubility": +1, "md_rmsd": -1}  # +1: higher=better
y = (pool["truth"] == "good").astype(int).values
print("Single-metric separability (ROC-AUC, EXAMPLE_DATA — illustrative only):")
aucs = {}
for m, direction in metric_dir.items():
    s = pool[m].values.astype(float) * direction
    mask = ~np.isnan(s)
    if len(set(y[mask])) > 1:
        aucs[m] = roc_auc_score(y[mask], s[mask])
        print(f"  {m:18s} AUC={aucs[m]:.3f}  (N={mask.sum()})")
print("\nNo metric reaches 1.0 — each only partially separates good from bad.")

Single-metric separability (ROC-AUC, EXAMPLE_DATA — illustrative only):
  scrmsd             AUC=0.723  (N=200)
  plddt              AUC=0.724  (N=200)
  pae_interaction    AUC=0.722  (N=200)
  scrmsd_orthogonal  AUC=0.680  (N=200)
  solubility         AUC=0.767  (N=200)
  md_rmsd            AUC=0.790  (N=200)

No metric reaches 1.0 — each only partially separates good from bad.


In [31]:
fig, axes = plt.subplots(2, 3, figsize=(10, 5.5))
for ax, m in zip(axes.ravel(), metric_dir):
    g = pool.loc[pool["truth"] == "good", m].dropna()
    b = pool.loc[pool["truth"] == "bad", m].dropna()
    ax.hist(g, bins=20, alpha=0.6, label="good", density=True)
    ax.hist(b, bins=20, alpha=0.6, label="bad", density=True)
    ax.set_title(f"{m} (AUC={aucs.get(m, float('nan')):.2f})", fontsize=9)
    ax.legend(fontsize=7)
plt.suptitle("Metric distributions by planted truth — note the OVERLAP (EXAMPLE_DATA)")
plt.tight_layout(); plt.savefig("results/p05_metric_overlap.png", dpi=150); plt.show()
print("saved results/p05_metric_overlap.png  (SYNTHETIC EXAMPLE_DATA)")

saved results/p05_metric_overlap.png  (SYNTHETIC EXAMPLE_DATA)


## 3 · Cutoff-sensitivity sweep

How brittle is the filter to its cutoffs? Sweep the Layer-1 `scrmsd` and `plddt` cutoffs over a
sensible range and watch **survival count** and **enrichment** move. Where the surface is flat, your
choice is safe; where it is steep, your short-list depends on an arbitrary threshold — say so.

In [32]:
mono = pool[pool["design_type"] == "monomer"].copy()
yk = (mono["truth"] == "good").astype(int).values

scrmsd_grid = np.round(np.arange(1.5, 3.01, 0.25), 2)
plddt_grid = np.arange(70, 91, 5)

surv_grid = np.zeros((len(plddt_grid), len(scrmsd_grid)))
prec_grid = np.zeros_like(surv_grid)
for i, pl in enumerate(plddt_grid):
    for j, sc in enumerate(scrmsd_grid):
        keep = (mono["scrmsd"] <= sc) & (mono["plddt"] >= pl)
        n = int(keep.sum())
        surv_grid[i, j] = n
        prec_grid[i, j] = (yk[keep.values].mean() if n else np.nan)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
im0 = ax[0].imshow(surv_grid, origin="lower", aspect="auto", cmap="viridis")
ax[0].set_title("survivors (monomers, EXAMPLE_DATA)")
im1 = ax[1].imshow(prec_grid, origin="lower", aspect="auto", cmap="magma", vmin=0, vmax=1)
ax[1].set_title("enrichment / precision (EXAMPLE_DATA)")
for a in ax:
    a.set_xticks(range(len(scrmsd_grid))); a.set_xticklabels(scrmsd_grid, fontsize=7)
    a.set_yticks(range(len(plddt_grid))); a.set_yticklabels(plddt_grid, fontsize=7)
    a.set_xlabel("scRMSD cutoff (<=)"); a.set_ylabel("pLDDT cutoff (>=)")
fig.colorbar(im0, ax=ax[0], fraction=0.046); fig.colorbar(im1, ax=ax[1], fraction=0.046)
plt.tight_layout(); plt.savefig("results/p05_cutoff_sensitivity.png", dpi=150); plt.show()
print("saved results/p05_cutoff_sensitivity.png  (SYNTHETIC EXAMPLE_DATA)")
print("Read it: tightening cutoffs raises enrichment but shrinks the survivor pool — the trade-off is the story.")

saved results/p05_cutoff_sensitivity.png  (SYNTHETIC EXAMPLE_DATA)
Read it: tightening cutoffs raises enrichment but shrinks the survivor pool — the trade-off is the story.


## 4 · Ranking stability under cutoff change `[extension]`

Does the **top-N** short-list change as cutoffs move? Compare the top-10 design IDs at a loose vs a
strict Layer-1 cutoff; the overlap (Jaccard) tells you how reproducible your short-list is. Low
overlap = your "best 10" is an artifact of an arbitrary threshold.

In [33]:
def top_ids(scrmsd_cut, plddt_cut, k=10):
    ds = []
    for _, r in mono.iterrows():
        d = row_to_design(r)
        ds.append(d)
    cut = dict(scrmsd=scrmsd_cut, plddt=plddt_cut, pae=None)
    for d in ds:
        fp.self_consistency(d, cut)
    ranked = fp.rank_designs(ds)
    return [d.design_id for d in ranked[:k]]

loose = set(top_ids(3.0, 70))
strict = set(top_ids(1.8, 88))
jac = len(loose & strict) / len(loose | strict)
print(f"top-10 overlap (Jaccard) loose vs strict cutoffs: {jac:.2f}")
print("loose-only :", sorted(loose - strict)[:5], "...")
print("strict-only:", sorted(strict - loose)[:5], "...")
print("Low overlap ⇒ the short-list is cutoff-sensitive; report this, do not hide it.")

top-10 overlap (Jaccard) loose vs strict cutoffs: 0.67
loose-only : ['EXAMPLE_DATA_0041', 'EXAMPLE_DATA_0160'] ...
strict-only: ['EXAMPLE_DATA_0071', 'EXAMPLE_DATA_0089'] ...
Low overlap ⇒ the short-list is cutoff-sensitive; report this, do not hide it.


## D3 (part 2) checklist
- [ ] Enrichment (precision) + recall per layer on the labeled pool; the gap from 1.0 discussed.
- [ ] Metric-overlap figure: no single metric separates good/bad (overlapping distributions).
- [ ] Cutoff-sensitivity sweep (survival + enrichment surfaces); brittle regions identified.
- [ ] Ranking-stability check (top-N overlap under cutoff change).
- [ ] Every figure labeled `EXAMPLE_DATA`; false positives among survivors stated explicitly.

**Next:** `05_validation_plan.ipynb` — optional Layer 4 (MD) hook, packaging, and cohort adoption.

---

## ▶︎ Section 6 / 6 — `05_validation_plan.ipynb`

---

# 05 · Validation plan — Layer 4 hook, packaging & cohort adoption

**Standard slot:** *validation plan.* **For Project 05 this means:** this project ships *software*, so
the "plan" is how the engine gets **adopted and maintained**: wire the optional **Layer 4 (short MD)**
hook, sketch a pip-installable package (stretch), and document **how the cohort proposes cutoff
changes via pull request** — never by silently overwriting the shared file (D4/D5).

Run `00`–`04` first.

## Setup paths

In [34]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

paths ready; cwd = /home/user/biofx_python/denovo_protein_design_course/projects/project_05_triage_pipeline/notebooks


## 1 · Optional **Layer 4 — dynamics (short MD)** hook

Static metrics can pass a design that *folds but melts*. Layer 4 runs a short MD and checks the
structure does not drift (`md_rmsd ≤ max_md_rmsd`). It is **expensive and optional** — at campaign
scale it is usually not worth it; reserve it for the final short-list. The planted
`BAD_L4_dynamics` design passes L1–L3 and only L4 catches it.

In [35]:
import filtering_pipeline as fp
import make_example_pool as mep

planted = {r["design_id"]: r for r in mep.known_designs()}
def design_from_row(row):
    drop = {"truth", "note", "design_id"}
    return fp.Design(design_id=row["design_id"], sequence="M",
                     **{k: v for k, v in row.items() if k not in drop})

cut = fp.DEFAULT_CUTOFFS["monomer"]
melt = design_from_row(planted["EXAMPLE_DATA_BAD_L4_dynamics"])
assert fp.self_consistency(melt, cut) is True, "folds-but-melts passes L1"
assert fp.physics_filter(melt, cut) is True, "...and L3"
assert fp.dynamics_filter(melt) is False, "...but L4 (MD drift) catches it"
print(f"BAD_L4_dynamics: md_rmsd={melt.md_rmsd}  -> L1/L3 pass, L4 fails  notes={melt.notes}")
print("Layer 4 verified. (Run the full pipeline with use_layers=(1,2,3,4) to include MD.)")

BAD_L4_dynamics: md_rmsd=4.5  -> L1/L3 pass, L4 fails  notes=['failed L4 dynamics']
Layer 4 verified. (Run the full pipeline with use_layers=(1,2,3,4) to include MD.)


### Where the MD numbers come from (behind the boundary)

The engine only *thresholds* `md_rmsd`. Producing it is an upstream OpenMM job — solvate, minimize,
short NVT/NPT run, measure mean backbone RMSD vs the design. That is GPU/CPU time you spend only on a
short-list. Pseudocode (implement on Colab with OpenMM; pin the version):

In [36]:
# Pseudocode — produce md_rmsd upstream, then set it on the Design (engine stays GPU-free):
#
#   from openmm.app import *; from openmm import *; from openmm.unit import *
#   # 1. load design PDB, add H, solvate, add ions
#   # 2. minimize; equilibrate; run 10-50 ns NVT/NPT
#   # 3. align each frame to frame 0; md_rmsd = mean backbone RMSD over the trajectory
#   design.md_rmsd = computed_md_rmsd
#   fp.dynamics_filter(design, max_md_rmsd=3.0)
print("MD is the optional, expensive layer — only worth it on the final short-list.")

MD is the optional, expensive layer — only worth it on the final short-list.


## 2 · Package as a pip-installable module `[stretch]`

To make the engine trivially adoptable, it can ship as a tiny package. Below is a **minimal
`pyproject.toml` scaffold shown in a cell** — do **not** commit it as a real package in this repo; the
canonical home of the engine is `shared/filtering_pipeline.py`. This is just to show the shape.

In [37]:
pyproject = """
[build-system]
requires = ["setuptools>=68", "wheel"]
build-backend = "setuptools.build_meta"

[project]
name = "denovo-filtering-pipeline"
version = "1.0.0"
description = "4-layer in-silico triage filter for de novo protein design (cohort shared engine)."
requires-python = ">=3.9"
dependencies = ["numpy", "pandas", "matplotlib", "biopython>=1.84"]

[project.optional-dependencies]
test = ["pytest"]
"""
print(pyproject)
print("# Scaffold only — the engine's canonical home is shared/filtering_pipeline.py.")
print("# If packaged: `pip install -e .` then `import filtering_pipeline`. Do NOT commit this here.")


[build-system]
requires = ["setuptools>=68", "wheel"]
build-backend = "setuptools.build_meta"

[project]
name = "denovo-filtering-pipeline"
version = "1.0.0"
description = "4-layer in-silico triage filter for de novo protein design (cohort shared engine)."
requires-python = ">=3.9"
dependencies = ["numpy", "pandas", "matplotlib", "biopython>=1.84"]

[project.optional-dependencies]
test = ["pytest"]

# Scaffold only — the engine's canonical home is shared/filtering_pipeline.py.
# If packaged: `pip install -e .` then `import filtering_pipeline`. Do NOT commit this here.


## 3 · How the cohort adopts cutoff changes (via PR, not silent overwrite)

Your calibration may justify changing `DEFAULT_CUTOFFS`. Because every project depends on this engine,
changes go through a **reviewed pull request** with evidence — never a notebook silently overwriting
the shared file. Below shows the current cutoffs and a *template* for proposing new ones; put the
proposal (with enrichment + N) in the PR description.

In [38]:
import filtering_pipeline as fp
print("CURRENT shared DEFAULT_CUTOFFS:")
for k, v in fp.DEFAULT_CUTOFFS.items():
    print(" ", k, v)

# Proposed change — fill from YOUR notebook-04 enrichment study, then open a PR. Example shape:
proposed = {
    # "monomer": dict(scrmsd=1.8, plddt=88, pae=None),   # evidence: enrichment X->Y at N=...
}
print("\nProposed (put these + the enrichment/N evidence in your PR description):")
print(proposed)
print("\nDo NOT write this back to shared/filtering_pipeline.py from here — open a reviewed PR.")

CURRENT shared DEFAULT_CUTOFFS:
  monomer {'scrmsd': 2.0, 'plddt': 85, 'pae': None}
  binder {'scrmsd': 2.5, 'plddt': 80, 'pae': 10, 'rosetta_dG': -30, 'sc': 0.6}
  enzyme {'scrmsd': 2.0, 'plddt': 85, 'plddt_cat': 90, 'cat_geom': 0.5}
  antibody {'scrmsd': 3.0, 'plddt': 70, 'pae': 12}
  oligomer {'scrmsd': 2.5, 'plddt': 80, 'pae': 10}

Proposed (put these + the enrichment/N evidence in your PR description):
{}

Do NOT write this back to shared/filtering_pipeline.py from here — open a reviewed PR.


## 4 · The cohort-adoption guide (what downstream projects need)

Write a short "how to use this engine" page for Projects 06–25:
- **Import:** `import filtering_pipeline as fp` (after the Setup-paths cell).
- **Build:** one `fp.Design` per candidate; set the metrics your predictors produced.
- **Run:** `fp.run_pipeline(designs, design_type=..., use_layers=(1,2,3))`.
- **Report:** `fp.report(df, save_prefix="results/<proj>")` → ranked CSV + survival figure.
- **Controls every project must include:** a known-good positive control + a negative control
  (scrambled interface / catalytic dead-mutant) in their pool, so the funnel is interpretable.
- **The honest caveat to repeat in every report:** the filter *enriches*, it does not *guarantee*;
  report false positives among survivors and your hit rate, not the cherry.

## D4 / D5 checklist
- [ ] Optional Layer 4 (short MD) hook verified on the planted folds-but-melts design.
- [ ] `pyproject.toml` packaging scaffold shown (not committed as a real package).
- [ ] Cohort-adoption guide written; cutoff changes routed through reviewed PRs (no silent overwrite).
- [ ] Thesis chapter + 15-min talk + `v1.0` tagged release.
- [ ] Engine landed as `shared/filtering_pipeline.py` — Projects 01–25 now stand on it.

You're done — and the cohort now has a tested, documented triage engine.